# 06 グループワーク — AI カメラ活用アイデアソン

> **このノートの使い方**：セルを選んで **Shift + Enter** で実行します。上から順に進めてください。
> 解説（この文章）→ コード → 結果（画像）→ ✅ チェックポイント、の順で進みます。
> 🤖 **困ったら ChatGPT に聞く**（エラーは*全文*を貼る／コードの意味も気軽に質問）。TA への挙手もOK。

> ### ⚠️ 先に AI カメラのモデルを入れてください
> このノートは **AI カメラ用モデル `imx500-all`** が必要です。**Pi のターミナル**（SSH）で一度だけ実行します（約1分・オンライン）:
>
> ```bash
> sudo apt install -y imx500-all
> ```
>
> - もし `Firmware file /usr/share/imx500-models/....rpk does not exist.` と出たら、**これが未導入**という意味です。上のコマンドで入れてから、このノートを上から実行し直してください。
> - この Pi は**再起動すると配布時の状態に戻る**ため、再起動後はもう一度このコマンドが必要です。

**目標**：午前〜04 で身につけた「OpenCV ＋ AI カメラ」を土台に、**チームで****使えるモデルを選び・試し・現場での活用アイデアを練る**。余裕があれば**動くプロトタイプ**まで。

**進め方**：① 使えるモデルを見る → ② モデルを差し替えて試す → ③ 現場での活用アイデアを練る → ④（余裕があれば）最小デモを実装 → ⑤ 共有。

> **役割を緩く決めると進みます**：ドライバ（操作）／アイデア係／モデル係／発表係（兼任OK）。

🤖 **ChatGPT をフル活用**してOK。アイデア出しもコードも、どんどん相談しましょう。

## 6-1. 使えるモデルを見る
IMX500 には物体検出以外のモデルも入っています。**用途に合うものを選ぶ**のが第一歩。班の興味で選びましょう。

In [ ]:
import glob, os
for p in sorted(glob.glob('/usr/share/imx500-models/*.rpk')):
    print(os.path.basename(p))

| モデル（種類） | できること | 向いている課題の例 |
|---|---|---|
| **物体検出**(ssd/efficientdet) | 何が・どこに（枠＋ラベル、90種） | 在庫・人・車・備品のカウント |
| **姿勢推定**(posenet/hrnet) | 人の関節点 | 転倒検知・作業姿勢・スポーツ |
| **領域分割**(deeplab) | 画素単位で領域分け | 占有面積・はみ出し・経路 |
| **画像分類**(efficientnet) | 1枚が何か | 良品/不良・種類の仕分け |

> **モデルの広げ方**：本講座は同梱モデルから選ぶのが基本ですが、IMX500 は**学習済みモデルを変換して載せる**こともできます（Sony の AITRIOS／model export）。「こんなモデルがあれば現場で効く」「どんなデータで学習させたいか」まで言語化できれば上出来です。

## 6-2. モデルを差し替えて検出する（同じコードで）
下の `MODEL` を切り替えて、検出系モデルを試せます。**変えたらこのセルを再実行**（モデルをセンサへ転送するので初回は数十秒）。

In [ ]:
import cv2, time
import matplotlib.pyplot as plt
plt.rcParams['font.family'] = 'Noto Sans CJK JP'  # 日本語ラベルの豆腐対策（要 fonts-noto-cjk）
from IPython.display import clear_output
%matplotlib inline
from picamera2 import Picamera2
from picamera2.devices import IMX500

# ▼ ここを切り替える（どちらも同じコードで動く検出モデル）
MODEL = '/usr/share/imx500-models/imx500_network_ssd_mobilenetv2_fpnlite_320x320_pp.rpk'
# MODEL = '/usr/share/imx500-models/imx500_network_efficientdet_lite0_pp.rpk'

imx = IMX500(MODEL); labels = imx.network_intrinsics.labels
picam = Picamera2(imx.camera_num)
picam.configure(picam.create_preview_configuration(
    main={'size':(640,480),'format':'RGB888'}, controls={'FrameRate':15}, buffer_count=8))
imx.show_network_fw_progress_bar(); picam.start(); time.sleep(2)
print('モデル:', MODEL.split('/')[-1])

In [ ]:
THRESH = 0.4
for i in range(40):
    m = picam.capture_metadata(); outputs = imx.get_outputs(m, add_batch=True)
    frame = picam.capture_array()
    if outputs is not None:
        boxes, scores, classes = outputs[0][0], outputs[1][0], outputs[2][0]
        for box, score, cls in zip(boxes, scores, classes):
            if score < THRESH: continue
            x,y,w,h = imx.convert_inference_coords(box, m, picam)
            cv2.rectangle(frame,(x,y),(x+w,y+h),(0,255,0),2)
            cv2.putText(frame,f'{labels[int(cls)]} {score:.2f}',(x,y-6),
                        cv2.FONT_HERSHEY_SIMPLEX,0.5,(0,0,255),1)
    clear_output(wait=True)
    plt.figure(figsize=(7,5)); plt.imshow(cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)); plt.axis('off')
    plt.title(f'detect {i+1}/40'); plt.show()
print('終了')

> **姿勢推定・領域分割**は出力の形が違うため、まずは
> 端末（モニタ接続時）の 1 行デモで雰囲気を掴むのが早いです：
> ```
> rpicam-hello -t 10000 --post-process-file /usr/share/rpi-camera-assets/imx500_posenet.json
> ```

## 6-3. アイデアを練る（班で 1 枚）
選んだモデルを動かしながら、**現場での使い道**を発散します。下のテンプレを、班で話しながら埋めましょう（Markdown セルです。ダブルクリックで編集 → Shift+Enter）。

### 🧩 私たちの班のアイデア
- **誰の・どんな現場？**（例：店舗バックヤード／工場ライン／介護施設／駐車場）：
- **どんな課題？**（例：数えるのが手間／危険の見落とし／記録が残らない）：
- **どのモデルで何を検知？**（例：物体検出で「人」を数える）：
- **結果をどう活かす？**（例：人数を CSV 記録／しきい値で通知／領域だけぼかす）：
- **エッジ AI の利点が効く所**（低遅延／省帯域／プライバシー）：

> 🤖 **ChatGPT に聞いてみよう**
> 「ラズパイの AI カメラ（オンセンサー物体検出／姿勢推定）を使った、＿＿＿業の業務改善アイデアを 5 つ。各案『検知するもの／活かし方／嬉しさ』を 1 行で」。→ 出た案から、班で**現実的に効く 1 つ**に絞る。

## 6-4. （余裕があれば）最小デモを実装
「動く 1 機能」を目指します。例として**人数カウンタ**の出発点を置きました（6-2 を実行済み前提）。

In [ ]:
# 人数カウンタの出発点：person(=class 0) を数えて表示
import cv2
from IPython.display import clear_output
import matplotlib.pyplot as plt
plt.rcParams['font.family'] = 'Noto Sans CJK JP'  # 日本語ラベルの豆腐対策（要 fonts-noto-cjk）
for i in range(40):
    m = picam.capture_metadata(); outputs = imx.get_outputs(m, add_batch=True)
    frame = picam.capture_array(); count = 0
    if outputs is not None:
        boxes, scores, classes = outputs[0][0], outputs[1][0], outputs[2][0]
        for box, score, cls in zip(boxes, scores, classes):
            if score < 0.4 or labels[int(cls)] != 'person': continue
            x,y,w,h = imx.convert_inference_coords(box, m, picam)
            cv2.rectangle(frame,(x,y),(x+w,y+h),(0,255,0),2); count += 1
    cv2.putText(frame, f'people: {count}', (10,30), cv2.FONT_HERSHEY_SIMPLEX, 1.0, (0,0,255), 2)
    clear_output(wait=True)
    plt.figure(figsize=(7,5)); plt.imshow(cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)); plt.axis('off'); plt.show()
print('終了')

> 🤖 **ChatGPT に聞いてみよう**
> この人数カウンタを改造しよう。例：「検出した person の数が変わった時刻だけ log.csv に追記したい。今のコードに追加して」。**出たコードはまず動かして確認**。

## 6-5. 共有・発表（各班 1–2 分）
発表に入れる 3 点：
- **誰の何の課題**を、**どのモデル**で解くか
- なぜ**カメラの中の AI（エッジ AI）**が効くのか
- 作れたデモ（あれば画面共有）／次に作るなら何か

> **今日の一番大事な 2 つ**：(1) 画像は numpy 配列であり、画像処理は配列の変換である。(2) ルールベース(OpenCV)と学習ベース(AIカメラ)は**役割分担して組み合わせる**。

## 6-9. 後始末（必ず実行）

In [ ]:
picam.close(); print('お疲れさまでした！ 発表の準備を。')